# Amazon Polarity Fine-Tuning Experiments

This notebook is organized around **three data splits** and **three training approaches**:

- **Data splits:** 1%, 5%, and 10% of the training data
- **Training methods:** Head-only training, LoRA with ranks `r=4, 8, 16`, and full fine-tuning

## Table of Contents

1. Environment setup and GPU checks
2. Dataset loading and inspection
3. Train/validation/test split creation
4. Tokenization and batching
5. Shared training utilities
6. Debug run for verifying the training pipeline
7. 1% data split experiments
   - Head-only
   - LoRA `r=8` initial run
   - Full fine-tuning
   - LoRA rank sweep: `r=4, 8, 16`
8. 5% data split experiments
   - Head-only
   - LoRA rank sweep: `r=4, 8, 16`
   - Full fine-tuning
9. 10% data split experiments
   - Head-only
   - LoRA rank sweep: `r=4, 8, 16`
   - Full fine-tuning
10. Save final summaries to Google Drive

> Run the notebook from top to bottom because later experiment sections reuse helper functions, tokenized datasets, and summary variables created earlier.


# 1. Environment setup and GPU checks

In [ ]:
# Core libraries
!pip -q install -U torch torchvision torchaudio
!pip -q install -U transformers datasets evaluate peft accelerate
!pip -q install -U scikit-learn pandas matplotlib tqdm

# Optional but important for your proposal
!pip -q install -U bitsandbytes


In [ ]:
import sys
import os
import platform
import importlib

print("Python version:", sys.version)
print("Platform:", platform.platform())
print("Processor:", platform.processor())


In [ ]:
libs = [
    "torch",
    "transformers",
    "datasets",
    "evaluate",
    "peft",
    "accelerate",
    "sklearn",
    "pandas",
    "matplotlib",
    "bitsandbytes"
]

for lib in libs:
    try:
        module = importlib.import_module(lib)
        version = getattr(module, "__version__", "version_not_found")
        print(f"{lib}: {version}")
    except Exception as e:
        print(f"{lib}: NOT AVAILABLE -> {e}")


In [ ]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA version (torch build):", torch.version.cuda)
    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Total VRAM: {props.total_memory / (1024**3):.2f} GB")
        print(f"  Multiprocessors: {props.multi_processor_count}")
        print(f"  Compute capability: {props.major}.{props.minor}")

    # bf16 support
    try:
        print("bf16 supported:", torch.cuda.is_bf16_supported())
    except Exception as e:
        print("bf16 supported: could not determine ->", e)
else:
    print("No CUDA GPU detected.")


In [ ]:
try:
    import bitsandbytes as bnb
    print("bitsandbytes imported successfully")
    print("bitsandbytes version:", bnb.__version__)
except Exception as e:
    print("bitsandbytes import FAILED")
    print("Error:", e)


In [ ]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print("Seed set to:", SEED)
print("Environment sanity step completed.")


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    device = "cuda"
    x = torch.randn(1024, 1024, device=device)
    y = torch.randn(1024, 1024, device=device)

    try:
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            z = x @ y
        print("fp16 autocast: AVAILABLE")
        print("Output dtype:", z.dtype)
    except Exception as e:
        print("fp16 autocast: NOT AVAILABLE")
        print("Error:", e)
else:
    print("No CUDA GPU found")


In [ ]:
import torch

if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    print("Compute capability:", f"{major}.{minor}")


# 2. Dataset Loading and Inspection

Load the Amazon Polarity dataset, inspect the available splits and columns, and combine `title` + `content` into one text field for classification.


In [ ]:
from datasets import load_dataset

dataset = load_dataset("fancyzhx/amazon_polarity")
dataset


In [ ]:
print(dataset)

for split in dataset.keys():
    print(f"{split}: {len(dataset[split])}")


In [ ]:
print("Train columns:", dataset["train"].column_names)
print()
print("First training example:")
print(dataset["train"][0])


In [ ]:
train_labels = dataset["train"]["label"]

unique_labels = sorted(set(train_labels))
print("Unique labels:", unique_labels)

from collections import Counter
print("Label counts (train sample check):")
print(Counter(train_labels[:10000]))


In [ ]:
sample = dataset["train"][0]

print("Available keys:", list(sample.keys()))
print("\nTITLE:\n", sample.get("title", "N/A"))
print("\nCONTENT:\n", sample.get("content", "N/A"))
print("\nLABEL:\n", sample.get("label", "N/A"))


In [ ]:
def combine_text(example):
    title = example["title"].strip() if example["title"] else ""
    content = example["content"].strip() if example["content"] else ""
    example["text"] = (title + " " + content).strip()
    return example

dataset = dataset.map(combine_text)

print(dataset["train"][0]["text"][:500])
print("\nLabel:", dataset["train"][0]["label"])


In [ ]:
print("Updated train columns:", dataset["train"].column_names)

empty_count = sum(1 for x in dataset["train"].select(range(1000)) if len(x["text"].strip()) == 0)
print("Empty texts in first 1000 train samples:", empty_count)

text_lengths = [len(dataset["train"][i]["text"]) for i in range(10)]
print("Character lengths of first 10 texts:", text_lengths)


# 3. Train/Validation/Test Split Creation

Create a held-out validation set and then define the 1%, 5%, and 10% training subsets used for the project experiments.


In [ ]:
SEED = 42

train_dataset = dataset["train"].shuffle(seed=SEED)
test_dataset = dataset["test"]

print(train_dataset)
print(test_dataset)


In [ ]:
VAL_SIZE = 50_000

val_full = train_dataset.select(range(VAL_SIZE))
train_full = train_dataset.select(range(VAL_SIZE, len(train_dataset)))
test_full = test_dataset

print("train_full:", len(train_full))
print("val_full:", len(val_full))
print("test_full:", len(test_full))


In [ ]:
from collections import Counter

def label_stats(ds, name, n=10000):
    labels = ds.select(range(min(n, len(ds))))["label"]
    counts = Counter(labels)
    total = sum(counts.values())
    props = {k: round(v / total, 4) for k, v in counts.items()}
    print(f"{name} label counts (first {min(n, len(ds))} rows): {counts}")
    print(f"{name} label proportions: {props}")

label_stats(train_full, "train_full")
label_stats(val_full, "val_full")
label_stats(test_full, "test_full")


In [ ]:
train_1pct = train_full.select(range(int(0.01 * len(train_full))))
train_5pct = train_full.select(range(int(0.05 * len(train_full))))
train_10pct = train_full.select(range(int(0.10 * len(train_full))))

print("train_1pct:", len(train_1pct))
print("train_5pct:", len(train_5pct))
print("train_10pct:", len(train_10pct))


In [ ]:
train_debug = train_full.select(range(5000))
val_debug = val_full.select(range(1000))

print("train_debug:", len(train_debug))
print("val_debug:", len(val_debug))


In [ ]:
data_splits = {
    "train_full": train_full,
    "val_full": val_full,
    "test_full": test_full,
    "train_1pct": train_1pct,
    "train_5pct": train_5pct,
    "train_10pct": train_10pct,
    "train_debug": train_debug,
    "val_debug": val_debug,
}

for k, v in data_splits.items():
    print(k, len(v))


# 4. Tokenization and Batching

Load the DistilBERT tokenizer, tokenize the training and validation datasets, and set up dynamic padding with a data collator.


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded:", MODEL_NAME)
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Model max length reported by tokenizer:", tokenizer.model_max_length)


In [ ]:
sample_text = train_debug[0]["text"]

encoded_sample = tokenizer(
    sample_text,
    truncation=True,
    max_length=MAX_LENGTH,
    padding=False
)

print("Original text (first 300 chars):")
print(sample_text[:300])

print("\nTokenized keys:", encoded_sample.keys())
print("Number of input_ids:", len(encoded_sample["input_ids"]))
print("First 20 token ids:", encoded_sample["input_ids"][:20])
print("Attention mask length:", len(encoded_sample["attention_mask"]))


In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False   # dynamic padding later in the data collator
    )


In [ ]:
tokenized_train_debug = train_debug.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing train_debug"
)

tokenized_val_debug = val_debug.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing val_debug"
)

print(tokenized_train_debug)
print(tokenized_val_debug)


In [ ]:
tokenized_train_debug = tokenized_train_debug.rename_column("label", "labels")
tokenized_val_debug = tokenized_val_debug.rename_column("label", "labels")

keep_cols = ["input_ids", "attention_mask", "labels"]

tokenized_train_debug = tokenized_train_debug.remove_columns(
    [col for col in tokenized_train_debug.column_names if col not in keep_cols]
)

tokenized_val_debug = tokenized_val_debug.remove_columns(
    [col for col in tokenized_val_debug.column_names if col not in keep_cols]
)

print("Train debug columns:", tokenized_train_debug.column_names)
print("Val debug columns:", tokenized_val_debug.column_names)


In [ ]:
print("One tokenized training example:")
print(tokenized_train_debug[0])

print("\nLengths:")
print("input_ids length:", len(tokenized_train_debug[0]["input_ids"]))
print("attention_mask length:", len(tokenized_train_debug[0]["attention_mask"]))
print("label:", tokenized_train_debug[0]["labels"])


In [ ]:
lengths = [len(tokenized_train_debug[i]["input_ids"]) for i in range(20)]

print("First 20 tokenized lengths:")
print(lengths)

print("\nMax observed length in first 20:", max(lengths))
print("Configured MAX_LENGTH:", MAX_LENGTH)


In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(data_collator)


In [ ]:
batch = data_collator([
    tokenized_train_debug[i] for i in range(4)
])

print("Batch keys:", batch.keys())
print("input_ids shape:", batch["input_ids"].shape)
print("attention_mask shape:", batch["attention_mask"].shape)
print("labels shape:", batch["labels"].shape)
print("\nBatch input_ids:")
print(batch["input_ids"])


In [ ]:
val_5k = val_full.select(range(5000))

print("val_5k:", len(val_5k))


In [ ]:
tokenized_train_1pct = train_1pct.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing train_1pct"
)

tokenized_val_5k = val_5k.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing val_5k"
)

tokenized_train_1pct = tokenized_train_1pct.rename_column("label", "labels")
tokenized_val_5k = tokenized_val_5k.rename_column("label", "labels")

keep_cols = ["input_ids", "attention_mask", "labels"]

tokenized_train_1pct = tokenized_train_1pct.remove_columns(
    [col for col in tokenized_train_1pct.column_names if col not in keep_cols]
)

tokenized_val_5k = tokenized_val_5k.remove_columns(
    [col for col in tokenized_val_5k.column_names if col not in keep_cols]
)

print(tokenized_train_1pct)
print(tokenized_val_5k)


# 5. Shared Training Utilities

Define reusable metric functions, parameter-count helpers, and GPU memory tracking utilities used across all experiments.


In [ ]:
import time
import math
import numpy as np
import torch

from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification

def compute_classification_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")

    return {
        "accuracy": acc,
        "f1_macro": f1
    }


In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    pct = 100 * trainable / total
    return {
        "total_params": total,
        "trainable_params": trainable,
        "trainable_pct": pct
    }

def print_parameter_report(model, title="Model"):
    stats = count_parameters(model)
    print(f"{title} parameter report")
    print(f"  Total params     : {stats['total_params']:,}")
    print(f"  Trainable params : {stats['trainable_params']:,}")
    print(f"  Trainable %      : {stats['trainable_pct']:.4f}%")


In [ ]:
def reset_gpu_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def get_peak_gpu_memory_mb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return None


## 4.1 Model Sanity Check

Load a fresh DistilBERT sequence-classification model, freeze the encoder for head-only training, and verify a forward pass on a small batch.


In [ ]:
MODEL_NAME = "distilbert-base-uncased"
NUM_LABELS = 2

id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

head_only_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

print(head_only_model.__class__.__name__)
print_parameter_report(head_only_model, title="Fresh DistilBERT classifier")


In [ ]:
for param in head_only_model.parameters():
    param.requires_grad = False

for name, param in head_only_model.named_parameters():
    if name.startswith("pre_classifier") or name.startswith("classifier"):
        param.requires_grad = True

print_parameter_report(head_only_model, title="Head-only DistilBERT")


In [ ]:
trainable_names = [name for name, p in head_only_model.named_parameters() if p.requires_grad]

print("Trainable parameter tensors:")
for name in trainable_names:
    print(" -", name)

print("\nNumber of trainable tensors:", len(trainable_names))


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
head_only_model = head_only_model.to(device)

print("Using device:", device)


In [ ]:
small_batch = data_collator([tokenized_train_debug[i] for i in range(4)])

small_batch = {k: v.to(device) for k, v in small_batch.items()}

for k, v in small_batch.items():
    print(k, v.shape, v.dtype)


In [ ]:
head_only_model.eval()
reset_gpu_memory()

with torch.no_grad():
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        outputs = head_only_model(**small_batch)

print("Loss:", outputs.loss.item())
print("Logits shape:", outputs.logits.shape)
print("Logits:")
print(outputs.logits)

peak_mem = get_peak_gpu_memory_mb()
print("Peak GPU memory during forward pass (MB):", peak_mem)


In [ ]:
logits = outputs.logits.detach().cpu().numpy()
labels = small_batch["labels"].detach().cpu().numpy()
preds = np.argmax(logits, axis=-1)

print("Labels:", labels)
print("Preds :", preds)
print("Batch accuracy:", accuracy_score(labels, preds))
print("Batch macro-F1:", f1_score(labels, preds, average="macro"))


# 6. Debug Run

Run a small head-only training job on the debug subset to confirm that the training pipeline, metrics, callbacks, and memory tracking work before running the full experiments.


In [ ]:
from transformers import TrainingArguments, Trainer


In [ ]:
def build_head_only_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=id2label,
        label2id=label2id
    )

    for param in model.parameters():
        param.requires_grad = False

    for name, param in model.named_parameters():
        if name.startswith("pre_classifier") or name.startswith("classifier"):
            param.requires_grad = True

    return model

head_only_train_model = build_head_only_model()
print_parameter_report(head_only_train_model, title="Fresh head-only model for training")


In [ ]:
import inspect
from transformers import TrainingArguments

raw_args = dict(
    output_dir="./runs/head_only_debug",

    do_train=True,
    do_eval=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,

    num_train_epochs=2,
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,

    logging_strategy="steps",
    logging_steps=50,

    fp16=True,
    bf16=False,

    report_to="none",
    seed=SEED
)

sig = inspect.signature(TrainingArguments.__init__)
accepted = set(sig.parameters.keys())

filtered_args = {k: v for k, v in raw_args.items() if k in accepted}

print("Using these TrainingArguments keys:")
for k in filtered_args:
    print("-", k)

training_args_head_debug = TrainingArguments(**filtered_args)
print(training_args_head_debug)


In [ ]:
head_only_trainer = Trainer(
    model=head_only_train_model,
    args=training_args_head_debug,
    train_dataset=tokenized_train_debug,
    eval_dataset=tokenized_val_debug,
    data_collator=data_collator,
    compute_metrics=compute_classification_metrics,
)


In [ ]:
reset_gpu_memory()

start_time = time.time()
train_result = head_only_trainer.train()
end_time = time.time()

head_only_train_time_sec = end_time - start_time
head_only_peak_mem_mb = get_peak_gpu_memory_mb()

print("Training finished.")
print(f"Wall-clock training time (sec): {head_only_train_time_sec:.2f}")
print(f"Peak GPU memory during training (MB): {head_only_peak_mem_mb:.2f}")


In [ ]:
from transformers.utils.notebook import NotebookProgressCallback

removed = head_only_trainer.pop_callback(NotebookProgressCallback)
print("Removed callback:", removed)

head_only_eval = head_only_trainer.evaluate()

print("Head-only eval results:")
for k, v in head_only_eval.items():
    print(f"{k}: {v}")


In [ ]:
head_only_summary = {
    "method": "head_only_debug",
    "train_samples": len(tokenized_train_debug),
    "val_samples": len(tokenized_val_debug),
    "epochs": training_args_head_debug.num_train_epochs,
    "trainable_params": count_parameters(head_only_train_model)["trainable_params"],
    "trainable_pct": count_parameters(head_only_train_model)["trainable_pct"],
    "train_time_sec": head_only_train_time_sec,
    "peak_gpu_mem_mb": head_only_peak_mem_mb,
    "eval_accuracy": head_only_eval.get("eval_accuracy"),
    "eval_f1_macro": head_only_eval.get("eval_f1_macro"),
    "eval_loss": head_only_eval.get("eval_loss"),
}

print(head_only_summary)


In [ ]:
pred_output = head_only_trainer.predict(tokenized_val_debug.select(range(20)))

preds = np.argmax(pred_output.predictions, axis=-1)
labels = pred_output.label_ids

print("Preds :", preds.tolist())
print("Labels:", labels.tolist())
print("Accuracy on first 20:", accuracy_score(labels, preds))
print("Macro-F1 on first 20:", f1_score(labels, preds, average="macro"))


# 7. Experiments on 1% Data Split

This section runs the 1% dataset experiments for head-only training, LoRA, and full fine-tuning.

## 7.1 Head-Only Training on 1% Split

Only the classifier head is trainable; the DistilBERT encoder remains frozen.


In [ ]:
head_only_1pct_model = build_head_only_model()
print_parameter_report(head_only_1pct_model, title="Head-only model for 1pct run")


In [ ]:
import inspect
from transformers import TrainingArguments

raw_args_1pct = dict(
    output_dir="./runs/head_only_1pct",

    do_train=True,
    do_eval=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,

    num_train_epochs=2,
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,

    logging_strategy="steps",
    logging_steps=100,

    fp16=True,
    bf16=False,

    report_to="none",
    seed=SEED
)

sig = inspect.signature(TrainingArguments.__init__)
accepted = set(sig.parameters.keys())
filtered_args_1pct = {k: v for k, v in raw_args_1pct.items() if k in accepted}

training_args_head_1pct = TrainingArguments(**filtered_args_1pct)
print(training_args_head_1pct)


In [ ]:
head_only_1pct_trainer = Trainer(
    model=head_only_1pct_model,
    args=training_args_head_1pct,
    train_dataset=tokenized_train_1pct,
    eval_dataset=tokenized_val_5k,
    data_collator=data_collator,
    compute_metrics=compute_classification_metrics,
)


In [ ]:
from transformers.utils.notebook import NotebookProgressCallback

try:
    head_only_1pct_trainer.remove_callback(NotebookProgressCallback)
    print("NotebookProgressCallback removed")
except Exception as e:
    print("Callback removal note:", e)


In [ ]:
reset_gpu_memory()

start_time = time.time()
train_result_1pct = head_only_1pct_trainer.train()
end_time = time.time()

head_only_1pct_train_time_sec = end_time - start_time
head_only_1pct_peak_mem_mb = get_peak_gpu_memory_mb()

print("Training finished.")
print(f"Wall-clock training time (sec): {head_only_1pct_train_time_sec:.2f}")
print(f"Peak GPU memory during training (MB): {head_only_1pct_peak_mem_mb:.2f}")


In [ ]:
head_only_1pct_eval = head_only_1pct_trainer.evaluate()

print("Head-only 1pct eval results:")
for k, v in head_only_1pct_eval.items():
    print(f"{k}: {v}")


In [ ]:
head_only_1pct_summary = {
    "method": "head_only_1pct",
    "train_samples": len(tokenized_train_1pct),
    "val_samples": len(tokenized_val_5k),
    "epochs": training_args_head_1pct.num_train_epochs,
    "trainable_params": count_parameters(head_only_1pct_model)["trainable_params"],
    "trainable_pct": count_parameters(head_only_1pct_model)["trainable_pct"],
    "train_time_sec": head_only_1pct_train_time_sec,
    "peak_gpu_mem_mb": head_only_1pct_peak_mem_mb,
    "eval_accuracy": head_only_1pct_eval.get("eval_accuracy"),
    "eval_f1_macro": head_only_1pct_eval.get("eval_f1_macro"),
    "eval_loss": head_only_1pct_eval.get("eval_loss"),
}

print(head_only_1pct_summary)


## 6.2 LoRA Setup and Initial `r=8` Run on 1% Split

Configure LoRA for DistilBERT attention projections and run the first LoRA experiment using rank `r=8`.


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType


In [ ]:
base_tmp = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

interesting = []
for name, module in base_tmp.named_modules():
    if any(x in name for x in ["attention", "q_lin", "v_lin", "k_lin", "out_lin"]):
        interesting.append(name)

for name in interesting[:80]:
    print(name)

del base_tmp
torch.cuda.empty_cache()


In [ ]:
# Fix for ImportError: Found an incompatible version of torchao
!pip install --upgrade torchao -qq

def build_lora_model_qv_r8():
    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=id2label,
        label2id=label2id
    )

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        bias="none",
        target_modules=["q_lin", "v_lin"]
    )

    model = get_peft_model(base_model, lora_config)
    return model

lora_1pct_model = build_lora_model_qv_r8()
print(lora_1pct_model.__class__.__name__)


In [ ]:
print_parameter_report(lora_1pct_model, title="LoRA QV r=8 model")

if hasattr(lora_1pct_model, "print_trainable_parameters"):
    lora_1pct_model.print_trainable_parameters()


In [ ]:
trainable_names = [name for name, p in lora_1pct_model.named_parameters() if p.requires_grad]

print("First 40 trainable parameter tensors:")
for name in trainable_names[:40]:
    print(" -", name)

print("\nTotal trainable tensors:", len(trainable_names))


In [ ]:
import inspect
from transformers import TrainingArguments

raw_args_lora_1pct = dict(
    output_dir="./runs/lora_qv_r8_1pct",

    do_train=True,
    do_eval=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,

    num_train_epochs=2,
    learning_rate=1e-3,
    weight_decay=0.01,
    warmup_ratio=0.1,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,

    logging_strategy="steps",
    logging_steps=100,

    fp16=True,
    bf16=False,

    report_to="none",
    seed=SEED
)

sig = inspect.signature(TrainingArguments.__init__)
accepted = set(sig.parameters.keys())
filtered_args_lora_1pct = {k: v for k, v in raw_args_lora_1pct.items() if k in accepted}

training_args_lora_1pct = TrainingArguments(**filtered_args_lora_1pct)
print(training_args_lora_1pct)


In [ ]:
lora_1pct_trainer = Trainer(
    model=lora_1pct_model,
    args=training_args_lora_1pct,
    train_dataset=tokenized_train_1pct,
    eval_dataset=tokenized_val_5k,
    data_collator=data_collator,
    compute_metrics=compute_classification_metrics,
)


In [ ]:
from transformers.utils.notebook import NotebookProgressCallback

try:
    lora_1pct_trainer.remove_callback(NotebookProgressCallback)
    print("NotebookProgressCallback removed")
except Exception as e:
    print("Callback removal note:", e)


In [ ]:
reset_gpu_memory()

start_time = time.time()
train_result_lora_1pct = lora_1pct_trainer.train()
end_time = time.time()

lora_1pct_train_time_sec = end_time - start_time
lora_1pct_peak_mem_mb = get_peak_gpu_memory_mb()

print("Training finished.")
print(f"Wall-clock training time (sec): {lora_1pct_train_time_sec:.2f}")
print(f"Peak GPU memory during training (MB): {lora_1pct_peak_mem_mb:.2f}")


In [ ]:
lora_1pct_eval = lora_1pct_trainer.evaluate()

print("LoRA 1pct eval results:")
for k, v in lora_1pct_eval.items():
    print(f"{k}: {v}")


In [ ]:
lora_1pct_summary = {
    "method": "lora_qv_r8_1pct",
    "train_samples": len(tokenized_train_1pct),
    "val_samples": len(tokenized_val_5k),
    "epochs": training_args_lora_1pct.num_train_epochs,
    "trainable_params": count_parameters(lora_1pct_model)["trainable_params"],
    "trainable_pct": count_parameters(lora_1pct_model)["trainable_pct"],
    "train_time_sec": lora_1pct_train_time_sec,
    "peak_gpu_mem_mb": lora_1pct_peak_mem_mb,
    "eval_accuracy": lora_1pct_eval.get("eval_accuracy"),
    "eval_f1_macro": lora_1pct_eval.get("eval_f1_macro"),
    "eval_loss": lora_1pct_eval.get("eval_loss"),
}

print(lora_1pct_summary)


In [ ]:
comparison = {
    "head_only_f1": head_only_1pct_summary["eval_f1_macro"],
    "lora_f1": lora_1pct_summary["eval_f1_macro"],
    "head_only_acc": head_only_1pct_summary["eval_accuracy"],
    "lora_acc": lora_1pct_summary["eval_accuracy"],
    "head_only_time_sec": head_only_1pct_summary["train_time_sec"],
    "lora_time_sec": lora_1pct_summary["train_time_sec"],
    "head_only_peak_mem_mb": head_only_1pct_summary["peak_gpu_mem_mb"],
    "lora_peak_mem_mb": lora_1pct_summary["peak_gpu_mem_mb"],
    "head_only_trainable_pct": head_only_1pct_summary["trainable_pct"],
    "lora_trainable_pct": lora_1pct_summary["trainable_pct"],
}

print(comparison)


## 6.3 Full Fine-Tuning on 1% Split

All DistilBERT parameters are trainable for the full fine-tuning baseline.


In [ ]:
def build_full_ft_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=id2label,
        label2id=label2id
    )
    return model

full_ft_1pct_model = build_full_ft_model()
print_parameter_report(full_ft_1pct_model, title="Full fine-tuning model for 1pct run")


In [ ]:
import inspect
from transformers import TrainingArguments

raw_args_full_ft_1pct = dict(
    output_dir="./runs/full_ft_1pct",

    do_train=True,
    do_eval=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,

    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,

    logging_strategy="steps",
    logging_steps=100,

    fp16=True,
    bf16=False,

    report_to="none",
    seed=SEED
)

sig = inspect.signature(TrainingArguments.__init__)
accepted = set(sig.parameters.keys())
filtered_args_full_ft_1pct = {k: v for k, v in raw_args_full_ft_1pct.items() if k in accepted}

training_args_full_ft_1pct = TrainingArguments(**filtered_args_full_ft_1pct)
print(training_args_full_ft_1pct)


In [ ]:
full_ft_1pct_trainer = Trainer(
    model=full_ft_1pct_model,
    args=training_args_full_ft_1pct,
    train_dataset=tokenized_train_1pct,
    eval_dataset=tokenized_val_5k,
    data_collator=data_collator,
    compute_metrics=compute_classification_metrics,
)


In [ ]:
from transformers.utils.notebook import NotebookProgressCallback

try:
    full_ft_1pct_trainer.remove_callback(NotebookProgressCallback)
    print("NotebookProgressCallback removed")
except Exception as e:
    print("Callback removal note:", e)


In [ ]:
reset_gpu_memory()

start_time = time.time()
train_result_full_ft_1pct = full_ft_1pct_trainer.train()
end_time = time.time()

full_ft_1pct_train_time_sec = end_time - start_time
full_ft_1pct_peak_mem_mb = get_peak_gpu_memory_mb()

print("Training finished.")
print(f"Wall-clock training time (sec): {full_ft_1pct_train_time_sec:.2f}")
print(f"Peak GPU memory during training (MB): {full_ft_1pct_peak_mem_mb:.2f}")


In [ ]:
full_ft_1pct_eval = full_ft_1pct_trainer.evaluate()

print("Full FT 1pct eval results:")
for k, v in full_ft_1pct_eval.items():
    print(f"{k}: {v}")


In [ ]:
full_ft_1pct_summary = {
    "method": "full_ft_1pct",
    "train_samples": len(tokenized_train_1pct),
    "val_samples": len(tokenized_val_5k),
    "epochs": training_args_full_ft_1pct.num_train_epochs,
    "trainable_params": count_parameters(full_ft_1pct_model)["trainable_params"],
    "trainable_pct": count_parameters(full_ft_1pct_model)["trainable_pct"],
    "train_time_sec": full_ft_1pct_train_time_sec,
    "peak_gpu_mem_mb": full_ft_1pct_peak_mem_mb,
    "eval_accuracy": full_ft_1pct_eval.get("eval_accuracy"),
    "eval_f1_macro": full_ft_1pct_eval.get("eval_f1_macro"),
    "eval_loss": full_ft_1pct_eval.get("eval_loss"),
}

print(full_ft_1pct_summary)


In [ ]:
all_three_comparison = {
    "head_only_f1": head_only_1pct_summary["eval_f1_macro"],
    "lora_f1": lora_1pct_summary["eval_f1_macro"],
    "full_ft_f1": full_ft_1pct_summary["eval_f1_macro"],

    "head_only_acc": head_only_1pct_summary["eval_accuracy"],
    "lora_acc": lora_1pct_summary["eval_accuracy"],
    "full_ft_acc": full_ft_1pct_summary["eval_accuracy"],

    "head_only_time_sec": head_only_1pct_summary["train_time_sec"],
    "lora_time_sec": lora_1pct_summary["train_time_sec"],
    "full_ft_time_sec": full_ft_1pct_summary["train_time_sec"],

    "head_only_peak_mem_mb": head_only_1pct_summary["peak_gpu_mem_mb"],
    "lora_peak_mem_mb": lora_1pct_summary["peak_gpu_mem_mb"],
    "full_ft_peak_mem_mb": full_ft_1pct_summary["peak_gpu_mem_mb"],

    "head_only_trainable_pct": head_only_1pct_summary["trainable_pct"],
    "lora_trainable_pct": lora_1pct_summary["trainable_pct"],
    "full_ft_trainable_pct": full_ft_1pct_summary["trainable_pct"],
}

print(all_three_comparison)


## 6.4 LoRA Rank Sweep on 1% Split: `r=4, 8, 16`

Define a reusable LoRA experiment function and compare LoRA ranks `4`, `8`, and `16` on the 1% split.


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

def build_lora_model(rank, alpha=None, dropout=0.1):
    if alpha is None:
        alpha = rank * 2  # simple practical default

    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=id2label,
        label2id=label2id
    )

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=rank,
        lora_alpha=alpha,
        lora_dropout=dropout,
        bias="none",
        target_modules=["q_lin", "v_lin"]
    )

    model = get_peft_model(base_model, lora_config)
    return model


In [ ]:
import inspect
from transformers import TrainingArguments

def build_training_args(output_dir, learning_rate=1e-3, epochs=2):
    raw_args = dict(
        output_dir=output_dir,

        do_train=True,
        do_eval=True,

        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=1,

        num_train_epochs=epochs,
        learning_rate=learning_rate,
        weight_decay=0.01,
        warmup_ratio=0.1,

        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,

        logging_strategy="steps",
        logging_steps=100,

        fp16=True,
        bf16=False,

        report_to="none",
        seed=SEED
    )

    sig = inspect.signature(TrainingArguments.__init__)
    accepted = set(sig.parameters.keys())
    filtered_args = {k: v for k, v in raw_args.items() if k in accepted}

    return TrainingArguments(**filtered_args)


In [ ]:
from transformers import Trainer
from transformers.utils.notebook import NotebookProgressCallback
import time

def run_lora_rank_experiment(rank, train_dataset, eval_dataset, split_tag="1pct", learning_rate=1e-3, epochs=2):
    print(f"\n===== Running LoRA rank {rank} =====")

    model = build_lora_model(rank=rank)
    print_parameter_report(model, title=f"LoRA rank {rank}")

    if hasattr(model, "print_trainable_parameters"):
        model.print_trainable_parameters()

    args = build_training_args(
        output_dir=f"./runs/lora_qv_r{rank}_{split_tag}",
        learning_rate=learning_rate,
        epochs=epochs
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        compute_metrics=compute_classification_metrics,
    )

    try:
        trainer.remove_callback(NotebookProgressCallback)
        print("NotebookProgressCallback removed")
    except Exception as e:
        print("Callback removal note:", e)

    reset_gpu_memory()

    start_time = time.time()
    trainer.train()
    end_time = time.time()

    train_time_sec = end_time - start_time
    peak_mem_mb = get_peak_gpu_memory_mb()

    eval_results = trainer.evaluate()

    summary = {
        "method": f"lora_qv_r{rank}_{split_tag}",
        "rank": rank,
        "train_samples": len(train_dataset),
        "val_samples": len(eval_dataset),
        "epochs": args.num_train_epochs,
        "trainable_params": count_parameters(model)["trainable_params"],
        "trainable_pct": count_parameters(model)["trainable_pct"],
        "train_time_sec": train_time_sec,
        "peak_gpu_mem_mb": peak_mem_mb,
        "eval_accuracy": eval_results.get("eval_accuracy"),
        "eval_f1_macro": eval_results.get("eval_f1_macro"),
        "eval_loss": eval_results.get("eval_loss"),
    }

    print("\nSummary:")
    print(summary)

    return summary


In [ ]:
lora_r4_summary = run_lora_rank_experiment(
    rank=4,
    train_dataset=tokenized_train_1pct,
    eval_dataset=tokenized_val_5k,
    learning_rate=1e-3,
    epochs=2
)


In [ ]:
lora_r16_summary = run_lora_rank_experiment(
    rank=16,
    train_dataset=tokenized_train_1pct,
    eval_dataset=tokenized_val_5k,
    learning_rate=1e-3,
    epochs=2
)


In [ ]:
lora_r8_summary = {
    "method": "lora_qv_r8_1pct",
    "rank": 8,
    "train_samples": lora_1pct_summary["train_samples"],
    "val_samples": lora_1pct_summary["val_samples"],
    "epochs": lora_1pct_summary["epochs"],
    "trainable_params": lora_1pct_summary["trainable_params"],
    "trainable_pct": lora_1pct_summary["trainable_pct"],
    "train_time_sec": lora_1pct_summary["train_time_sec"],
    "peak_gpu_mem_mb": lora_1pct_summary["peak_gpu_mem_mb"],
    "eval_accuracy": lora_1pct_summary["eval_accuracy"],
    "eval_f1_macro": lora_1pct_summary["eval_f1_macro"],
    "eval_loss": lora_1pct_summary["eval_loss"],
}

print(lora_r8_summary)


In [ ]:
import pandas as pd

lora_rank_df = pd.DataFrame([
    lora_r4_summary,
    lora_r8_summary,
    lora_r16_summary
]).sort_values("rank").reset_index(drop=True)

print(lora_rank_df)


In [ ]:
best_lora_row = lora_rank_df.sort_values("eval_f1_macro", ascending=False).iloc[0]

print("Best LoRA rank by macro-F1:")
print(best_lora_row)


In [ ]:
best_lora_vs_baselines = {
    "best_lora_rank": int(best_lora_row["rank"]),
    "best_lora_f1": float(best_lora_row["eval_f1_macro"]),
    "head_only_f1": head_only_1pct_summary["eval_f1_macro"],
    "full_ft_f1": full_ft_1pct_summary["eval_f1_macro"],

    "best_lora_acc": float(best_lora_row["eval_accuracy"]),
    "head_only_acc": head_only_1pct_summary["eval_accuracy"],
    "full_ft_acc": full_ft_1pct_summary["eval_accuracy"],

    "best_lora_time_sec": float(best_lora_row["train_time_sec"]),
    "head_only_time_sec": head_only_1pct_summary["train_time_sec"],
    "full_ft_time_sec": full_ft_1pct_summary["train_time_sec"],

    "best_lora_peak_mem_mb": float(best_lora_row["peak_gpu_mem_mb"]),
    "head_only_peak_mem_mb": head_only_1pct_summary["peak_gpu_mem_mb"],
    "full_ft_peak_mem_mb": full_ft_1pct_summary["peak_gpu_mem_mb"],
}

print(best_lora_vs_baselines)


# 8. Experiments on 5% Data Split

This section repeats the same training-method comparison on the 5% training subset.

## 8.1 Tokenize 5% Split and Run Head-Only Training


In [ ]:
tokenized_train_5pct = train_5pct.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing train_5pct"
)

tokenized_train_5pct = tokenized_train_5pct.rename_column("label", "labels")

keep_cols = ["input_ids", "attention_mask", "labels"]
tokenized_train_5pct = tokenized_train_5pct.remove_columns(
    [col for col in tokenized_train_5pct.column_names if col not in keep_cols]
)

print(tokenized_train_5pct)
print("train_5pct tokenized rows:", len(tokenized_train_5pct))
print("val_5k rows:", len(tokenized_val_5k))


In [ ]:
head_only_5pct_model = build_head_only_model()
print_parameter_report(head_only_5pct_model, title="Head-only model for 5pct run")


In [ ]:
training_args_head_5pct = build_training_args(
    output_dir="./runs/head_only_5pct",
    learning_rate=5e-4,
    epochs=2
)

print(training_args_head_5pct)


In [ ]:
head_only_5pct_trainer = Trainer(
    model=head_only_5pct_model,
    args=training_args_head_5pct,
    train_dataset=tokenized_train_5pct,
    eval_dataset=tokenized_val_5k,
    data_collator=data_collator,
    compute_metrics=compute_classification_metrics,
)

from transformers.utils.notebook import NotebookProgressCallback

try:
    head_only_5pct_trainer.remove_callback(NotebookProgressCallback)
    print("NotebookProgressCallback removed")
except Exception as e:
    print("Callback removal note:", e)


In [ ]:
reset_gpu_memory()

start_time = time.time()
head_only_5pct_trainer.train()
end_time = time.time()

head_only_5pct_train_time_sec = end_time - start_time
head_only_5pct_peak_mem_mb = get_peak_gpu_memory_mb()

print("Training finished.")
print(f"Wall-clock training time (sec): {head_only_5pct_train_time_sec:.2f}")
print(f"Peak GPU memory during training (MB): {head_only_5pct_peak_mem_mb:.2f}")


In [ ]:
head_only_5pct_eval = head_only_5pct_trainer.evaluate()

print("Head-only 5pct eval results:")
for k, v in head_only_5pct_eval.items():
    print(f"{k}: {v}")


In [ ]:
# Before using head_only_5pct_eval, ensure it's defined
# if 'head_only_5pct_eval' not in locals():
#     print("Warning: head_only_5pct_eval was not defined. Please run the previous cell (9_W_uvRn8u1N) to compute evaluation results.")
#     head_only_5pct_eval = {} # Provide a fallback empty dictionary

head_only_5pct_summary = {
    "method": "head_only_5pct",
    "train_samples": len(tokenized_train_5pct),
    "val_samples": len(tokenized_val_5k),
    "epochs": training_args_head_5pct.num_train_epochs,
    "trainable_params": count_parameters(head_only_5pct_model)["trainable_params"],
    "trainable_pct": count_parameters(head_only_5pct_model)["trainable_pct"],
    "train_time_sec": head_only_5pct_train_time_sec,
    "peak_gpu_mem_mb": head_only_5pct_peak_mem_mb,
    "eval_accuracy": head_only_5pct_eval.get("eval_accuracy"),
    "eval_f1_macro": head_only_5pct_eval.get("eval_f1_macro"),
    "eval_loss": head_only_5pct_eval.get("eval_loss"),
}

print(head_only_5pct_summary)


In [ ]:
head_only_scale_compare = {
    "head_only_1pct_f1": head_only_1pct_summary["eval_f1_macro"],
    "head_only_5pct_f1": head_only_5pct_summary["eval_f1_macro"],
    "head_only_1pct_acc": head_only_1pct_summary["eval_accuracy"],
    "head_only_5pct_acc": head_only_5pct_summary["eval_accuracy"],
    "head_only_1pct_time_sec": head_only_1pct_summary["train_time_sec"],
    "head_only_5pct_time_sec": head_only_5pct_summary["train_time_sec"],
    "head_only_1pct_peak_mem_mb": head_only_1pct_summary["peak_gpu_mem_mb"],
    "head_only_5pct_peak_mem_mb": head_only_5pct_summary["peak_gpu_mem_mb"],
}

print(head_only_scale_compare)


## 8.2 LoRA Training on 5% Split

Run LoRA experiments for ranks `r=4, 8, 16` and compare them against the 5% head-only baseline.


In [ ]:
lora_r16_5pct_summary = run_lora_rank_experiment(
    rank=16,
    train_dataset=tokenized_train_5pct,
    eval_dataset=tokenized_val_5k,
    split_tag="5pct",
    learning_rate=1e-3,
    epochs=2
)


In [ ]:
lora_r16_scale_compare = {
    "lora_r16_1pct_f1": lora_r16_summary["eval_f1_macro"],
    "lora_r16_5pct_f1": lora_r16_5pct_summary["eval_f1_macro"],
    "lora_r16_1pct_acc": lora_r16_summary["eval_accuracy"],
    "lora_r16_5pct_acc": lora_r16_5pct_summary["eval_accuracy"],
    "lora_r16_1pct_time_sec": lora_r16_summary["train_time_sec"],
    "lora_r16_5pct_time_sec": lora_r16_5pct_summary["train_time_sec"],
    "lora_r16_1pct_peak_mem_mb": lora_r16_summary["peak_gpu_mem_mb"],
    "lora_r16_5pct_peak_mem_mb": lora_r16_5pct_summary["peak_gpu_mem_mb"],
}

print(lora_r16_scale_compare)


In [ ]:
head_vs_lora_5pct = {
    "head_only_5pct_f1": head_only_5pct_summary["eval_f1_macro"],
    "lora_r16_5pct_f1": lora_r16_5pct_summary["eval_f1_macro"],
    "head_only_5pct_acc": head_only_5pct_summary["eval_accuracy"],
    "lora_r16_5pct_acc": lora_r16_5pct_summary["eval_accuracy"],
    "head_only_5pct_time_sec": head_only_5pct_summary["train_time_sec"],
    "lora_r16_5pct_time_sec": lora_r16_5pct_summary["train_time_sec"],
    "head_only_5pct_peak_mem_mb": head_only_5pct_summary["peak_gpu_mem_mb"],
    "lora_r16_5pct_peak_mem_mb": lora_r16_5pct_summary["peak_gpu_mem_mb"],
    "head_only_5pct_trainable_pct": head_only_5pct_summary["trainable_pct"],
    "lora_r16_5pct_trainable_pct": lora_r16_5pct_summary["trainable_pct"],
}

print(head_vs_lora_5pct)


## 8.3 Optional Intermediate Checkpoint Save

This optional checkpoint saves the summaries available so far. The final save cell at the end of the notebook writes the complete 1%, 5%, and 10% results.


In [ ]:
import json, os, torch
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/amazon-sentiment"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Save all result summaries
summaries = {}

# Add whichever of these exist — wrap each in try/except
for name, var in [
    ("head_only_1pct",      head_only_1pct_summary),
    ("head_only_5pct",      head_only_5pct_summary),
    ("lora_r4_1pct",        lora_r4_summary),
    ("lora_r8_1pct",        lora_r8_summary),
    ("lora_r16_1pct",       lora_r16_summary),
    ("lora_r16_5pct",       lora_r16_5pct_summary),
    ("full_ft_1pct",        full_ft_1pct_summary),
]:
    try:
        summaries[name] = var
    except NameError:
        pass

with open(f"{SAVE_DIR}/summaries.json", "w") as f:
    json.dump(summaries, f, indent=2)

print(f"Saved {len(summaries)} summaries:", list(summaries.keys()))

# 2. Save tokenized datasets (skip recomputation next session)
datasets_to_save = {}
for name, var in [
    ("tokenized_train_1pct",  tokenized_train_1pct),
    ("tokenized_train_5pct",  tokenized_train_5pct),
    ("tokenized_val_5k",      tokenized_val_5k),
]:
    try:
        datasets_to_save[name] = var
    except NameError:
        pass

torch.save(datasets_to_save, f"{SAVE_DIR}/tokenized_datasets.pt")
print(f"Saved datasets:", list(datasets_to_save.keys()))


In [ ]:
lora_r16_5pct_summary["method"] = "lora_qv_r16_5pct"
print(lora_r16_5pct_summary)


## 8.4 Complete 5% LoRA Rank Sweep and Comparison

Run the remaining 5% LoRA ranks and create summary comparison tables.


In [ ]:
lora_r4_5pct_summary = run_lora_rank_experiment(
    rank=4,
    train_dataset=tokenized_train_5pct,
    eval_dataset=tokenized_val_5k,
    split_tag="5pct",
    learning_rate=1e-3,
    epochs=2
)


In [ ]:
lora_r8_5pct_summary = run_lora_rank_experiment(
    rank=8,
    train_dataset=tokenized_train_5pct,
    eval_dataset=tokenized_val_5k,
    split_tag="5pct",
    learning_rate=1e-3,
    epochs=2
)


In [ ]:
import pandas as pd

lora_5pct_rank_df = pd.DataFrame([
    lora_r4_5pct_summary,
    lora_r8_5pct_summary,
    lora_r16_5pct_summary
]).sort_values("rank").reset_index(drop=True)

print(lora_5pct_rank_df)


In [ ]:
best_lora_5pct_row = lora_5pct_rank_df.sort_values("eval_f1_macro", ascending=False).iloc[0]

print("Best LoRA rank on 5% by macro-F1:")
print(best_lora_5pct_row)


In [ ]:
compare_head_vs_lora_5pct = pd.DataFrame([
    {
        "method": "head_only_5pct",
        "rank": None,
        "eval_accuracy": head_only_5pct_summary["eval_accuracy"],
        "eval_f1_macro": head_only_5pct_summary["eval_f1_macro"],
        "train_time_sec": head_only_5pct_summary["train_time_sec"],
        "peak_gpu_mem_mb": head_only_5pct_summary["peak_gpu_mem_mb"],
        "trainable_pct": head_only_5pct_summary["trainable_pct"],
    },
    {
        "method": lora_r4_5pct_summary["method"],
        "rank": lora_r4_5pct_summary["rank"],
        "eval_accuracy": lora_r4_5pct_summary["eval_accuracy"],
        "eval_f1_macro": lora_r4_5pct_summary["eval_f1_macro"],
        "train_time_sec": lora_r4_5pct_summary["train_time_sec"],
        "peak_gpu_mem_mb": lora_r4_5pct_summary["peak_gpu_mem_mb"],
        "trainable_pct": lora_r4_5pct_summary["trainable_pct"],
    },
    {
        "method": lora_r8_5pct_summary["method"],
        "rank": lora_r8_5pct_summary["rank"],
        "eval_accuracy": lora_r8_5pct_summary["eval_accuracy"],
        "eval_f1_macro": lora_r8_5pct_summary["eval_f1_macro"],
        "train_time_sec": lora_r8_5pct_summary["train_time_sec"],
        "peak_gpu_mem_mb": lora_r8_5pct_summary["peak_gpu_mem_mb"],
        "trainable_pct": lora_r8_5pct_summary["trainable_pct"],
    },
    {
        "method": lora_r16_5pct_summary["method"],
        "rank": lora_r16_5pct_summary["rank"],
        "eval_accuracy": lora_r16_5pct_summary["eval_accuracy"],
        "eval_f1_macro": lora_r16_5pct_summary["eval_f1_macro"],
        "train_time_sec": lora_r16_5pct_summary["train_time_sec"],
        "peak_gpu_mem_mb": lora_r16_5pct_summary["peak_gpu_mem_mb"],
        "trainable_pct": lora_r16_5pct_summary["trainable_pct"],
    },
])

print(compare_head_vs_lora_5pct)


## 8.5 Full Fine-Tuning on 5% Split

Run full fine-tuning on the 5% split and create the final 5% comparison table.


In [ ]:
full_ft_5pct_model = build_full_ft_model()
print_parameter_report(full_ft_5pct_model, title="Full fine-tuning model for 5pct run")


In [ ]:
import inspect
from transformers import TrainingArguments

raw_args_full_ft_5pct = dict(
    output_dir="./runs/full_ft_5pct",

    do_train=True,
    do_eval=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,

    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=64,

    logging_strategy="steps",
    logging_steps=100,

    fp16=True,
    bf16=False,

    report_to="none",
    seed=SEED
)

sig = inspect.signature(TrainingArguments.__init__)
accepted = set(sig.parameters.keys())
filtered_args_full_ft_5pct = {k: v for k, v in raw_args_full_ft_5pct.items() if k in accepted}

training_args_full_ft_5pct = TrainingArguments(**filtered_args_full_ft_5pct)
print(training_args_full_ft_5pct)


In [ ]:
full_ft_5pct_trainer = Trainer(
    model=full_ft_5pct_model,
    args=training_args_full_ft_5pct,
    train_dataset=tokenized_train_5pct,
    eval_dataset=tokenized_val_5k,
    data_collator=data_collator,
    compute_metrics=compute_classification_metrics,
)

from transformers.utils.notebook import NotebookProgressCallback

try:
    full_ft_5pct_trainer.remove_callback(NotebookProgressCallback)
    print("NotebookProgressCallback removed")
except Exception as e:
    print("Callback removal note:", e)


In [ ]:
reset_gpu_memory()

start_time = time.time()
full_ft_5pct_trainer.train()
end_time = time.time()

full_ft_5pct_train_time_sec = end_time - start_time
full_ft_5pct_peak_mem_mb = get_peak_gpu_memory_mb()

print("Training finished.")
print(f"Wall-clock training time (sec): {full_ft_5pct_train_time_sec:.2f}")
print(f"Peak GPU memory during training (MB): {full_ft_5pct_peak_mem_mb:.2f}")


In [ ]:
full_ft_5pct_eval = full_ft_5pct_trainer.evaluate()

print("Full FT 5pct eval results:")
for k, v in full_ft_5pct_eval.items():
    print(f"{k}: {v}")


In [ ]:
full_ft_5pct_summary = {
    "method": "full_ft_5pct",
    "train_samples": len(tokenized_train_5pct),
    "val_samples": len(tokenized_val_5k),
    "epochs": training_args_full_ft_5pct.num_train_epochs,
    "trainable_params": count_parameters(full_ft_5pct_model)["trainable_params"],
    "trainable_pct": count_parameters(full_ft_5pct_model)["trainable_pct"],
    "train_time_sec": full_ft_5pct_train_time_sec,
    "peak_gpu_mem_mb": full_ft_5pct_peak_mem_mb,
    "eval_accuracy": full_ft_5pct_eval.get("eval_accuracy"),
    "eval_f1_macro": full_ft_5pct_eval.get("eval_f1_macro"),
    "eval_loss": full_ft_5pct_eval.get("eval_loss"),
}

print(full_ft_5pct_summary)


In [ ]:
final_5pct_df = pd.DataFrame([
    {
        "method": "head_only_5pct",
        "rank": None,
        "eval_accuracy": head_only_5pct_summary["eval_accuracy"],
        "eval_f1_macro": head_only_5pct_summary["eval_f1_macro"],
        "train_time_sec": head_only_5pct_summary["train_time_sec"],
        "peak_gpu_mem_mb": head_only_5pct_summary["peak_gpu_mem_mb"],
        "trainable_pct": head_only_5pct_summary["trainable_pct"],
    },
    {
        "method": lora_r4_5pct_summary["method"],
        "rank": lora_r4_5pct_summary["rank"],
        "eval_accuracy": lora_r4_5pct_summary["eval_accuracy"],
        "eval_f1_macro": lora_r4_5pct_summary["eval_f1_macro"],
        "train_time_sec": lora_r4_5pct_summary["train_time_sec"],
        "peak_gpu_mem_mb": lora_r4_5pct_summary["peak_gpu_mem_mb"],
        "trainable_pct": lora_r4_5pct_summary["trainable_pct"],
    },
    {
        "method": lora_r8_5pct_summary["method"],
        "rank": lora_r8_5pct_summary["rank"],
        "eval_accuracy": lora_r8_5pct_summary["eval_accuracy"],
        "eval_f1_macro": lora_r8_5pct_summary["eval_f1_macro"],
        "train_time_sec": lora_r8_5pct_summary["train_time_sec"],
        "peak_gpu_mem_mb": lora_r8_5pct_summary["peak_gpu_mem_mb"],
        "trainable_pct": lora_r8_5pct_summary["trainable_pct"],
    },
    {
        "method": lora_r16_5pct_summary["method"],
        "rank": lora_r16_5pct_summary["rank"],
        "eval_accuracy": lora_r16_5pct_summary["eval_accuracy"],
        "eval_f1_macro": lora_r16_5pct_summary["eval_f1_macro"],
        "train_time_sec": lora_r16_5pct_summary["train_time_sec"],
        "peak_gpu_mem_mb": lora_r16_5pct_summary["peak_gpu_mem_mb"],
        "trainable_pct": lora_r16_5pct_summary["trainable_pct"],
    },
    {
        "method": full_ft_5pct_summary["method"],
        "rank": None,
        "eval_accuracy": full_ft_5pct_summary["eval_accuracy"],
        "eval_f1_macro": full_ft_5pct_summary["eval_f1_macro"],
        "train_time_sec": full_ft_5pct_summary["train_time_sec"],
        "peak_gpu_mem_mb": full_ft_5pct_summary["peak_gpu_mem_mb"],
        "trainable_pct": full_ft_5pct_summary["trainable_pct"],
    },
])

print(final_5pct_df)


# 9. Experiments on 10% Data Split

This section repeats the same training-method comparison on the 10% training subset.

## 9.1 Tokenize 10% Split and Run Head-Only Training


In [ ]:
tokenized_train_10pct = train_10pct.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing train_10pct"
)

tokenized_train_10pct = tokenized_train_10pct.rename_column("label", "labels")

keep_cols = ["input_ids", "attention_mask", "labels"]
tokenized_train_10pct = tokenized_train_10pct.remove_columns(
    [col for col in tokenized_train_10pct.column_names if col not in keep_cols]
)

print(tokenized_train_10pct)
print("train_10pct tokenized rows:", len(tokenized_train_10pct))
print("val_5k rows:", len(tokenized_val_5k))


In [ ]:
head_only_10pct_model = build_head_only_model()
print_parameter_report(head_only_10pct_model, title="Head-only model for 10pct run")


In [ ]:
training_args_head_10pct = build_training_args(
    output_dir="./runs/head_only_10pct",
    learning_rate=5e-4,
    epochs=2
)

print(training_args_head_10pct)


In [ ]:
head_only_10pct_trainer = Trainer(
    model=head_only_10pct_model,
    args=training_args_head_10pct,
    train_dataset=tokenized_train_10pct,
    eval_dataset=tokenized_val_5k,
    data_collator=data_collator,
    compute_metrics=compute_classification_metrics,
)

from transformers.utils.notebook import NotebookProgressCallback

try:
    head_only_10pct_trainer.remove_callback(NotebookProgressCallback)
    print("NotebookProgressCallback removed")
except Exception as e:
    print("Callback removal note:", e)


In [ ]:
reset_gpu_memory()

start_time = time.time()
head_only_10pct_trainer.train()
end_time = time.time()

head_only_10pct_train_time_sec = end_time - start_time
head_only_10pct_peak_mem_mb = get_peak_gpu_memory_mb()

print("Training finished.")
print(f"Wall-clock training time (sec): {head_only_10pct_train_time_sec:.2f}")
print(f"Peak GPU memory during training (MB): {head_only_10pct_peak_mem_mb:.2f}")


In [ ]:
head_only_10pct_eval = head_only_10pct_trainer.evaluate()

print("Head-only 10pct eval results:")
for k, v in head_only_10pct_eval.items():
    print(f"{k}: {v}")


In [ ]:
head_only_10pct_summary = {
    "method": "head_only_10pct",
    "train_samples": len(tokenized_train_10pct),
    "val_samples": len(tokenized_val_5k),
    "epochs": training_args_head_10pct.num_train_epochs,
    "trainable_params": count_parameters(head_only_10pct_model)["trainable_params"],
    "trainable_pct": count_parameters(head_only_10pct_model)["trainable_pct"],
    "train_time_sec": head_only_10pct_train_time_sec,
    "peak_gpu_mem_mb": head_only_10pct_peak_mem_mb,
    "eval_accuracy": head_only_10pct_eval.get("eval_accuracy"),
    "eval_f1_macro": head_only_10pct_eval.get("eval_f1_macro"),
    "eval_loss": head_only_10pct_eval.get("eval_loss"),
}

print(head_only_10pct_summary)


## 9.2 LoRA Rank Sweep on 10% Split: `r=4, 8, 16`

Run LoRA experiments for all three ranks on the 10% training subset and compare them with the 10% head-only result.


In [ ]:
lora_r4_10pct_summary = run_lora_rank_experiment(
    rank=4,
    train_dataset=tokenized_train_10pct,
    eval_dataset=tokenized_val_5k,
    split_tag="10pct",
    learning_rate=1e-3,
    epochs=2
)


In [ ]:
lora_r8_10pct_summary = run_lora_rank_experiment(
    rank=8,
    train_dataset=tokenized_train_10pct,
    eval_dataset=tokenized_val_5k,
    split_tag="10pct",
    learning_rate=1e-3,
    epochs=2
)


In [ ]:
lora_r16_10pct_summary = run_lora_rank_experiment(
    rank=16,
    train_dataset=tokenized_train_10pct,
    eval_dataset=tokenized_val_5k,
    split_tag="10pct",
    learning_rate=1e-3,
    epochs=2
)


In [ ]:
import pandas as pd

lora_10pct_rank_df = pd.DataFrame([
    lora_r4_10pct_summary,
    lora_r8_10pct_summary,
    lora_r16_10pct_summary
]).sort_values("rank").reset_index(drop=True)

print(lora_10pct_rank_df)


In [ ]:
best_lora_10pct_row = lora_10pct_rank_df.sort_values("eval_f1_macro", ascending=False).iloc[0]

print("Best LoRA rank on 10% by macro-F1:")
print(best_lora_10pct_row)


In [ ]:
compare_head_vs_lora_10pct = pd.DataFrame([
    {
        "method": "head_only_10pct",
        "rank": None,
        "eval_accuracy": head_only_10pct_summary["eval_accuracy"],
        "eval_f1_macro": head_only_10pct_summary["eval_f1_macro"],
        "train_time_sec": head_only_10pct_summary["train_time_sec"],
        "peak_gpu_mem_mb": head_only_10pct_summary["peak_gpu_mem_mb"],
        "trainable_pct": head_only_10pct_summary["trainable_pct"],
    },
    {
        "method": lora_r4_10pct_summary["method"],
        "rank": lora_r4_10pct_summary["rank"],
        "eval_accuracy": lora_r4_10pct_summary["eval_accuracy"],
        "eval_f1_macro": lora_r4_10pct_summary["eval_f1_macro"],
        "train_time_sec": lora_r4_10pct_summary["train_time_sec"],
        "peak_gpu_mem_mb": lora_r4_10pct_summary["peak_gpu_mem_mb"],
        "trainable_pct": lora_r4_10pct_summary["trainable_pct"],
    },
    {
        "method": lora_r8_10pct_summary["method"],
        "rank": lora_r8_10pct_summary["rank"],
        "eval_accuracy": lora_r8_10pct_summary["eval_accuracy"],
        "eval_f1_macro": lora_r8_10pct_summary["eval_f1_macro"],
        "train_time_sec": lora_r8_10pct_summary["train_time_sec"],
        "peak_gpu_mem_mb": lora_r8_10pct_summary["peak_gpu_mem_mb"],
        "trainable_pct": lora_r8_10pct_summary["trainable_pct"],
    },
    {
        "method": lora_r16_10pct_summary["method"],
        "rank": lora_r16_10pct_summary["rank"],
        "eval_accuracy": lora_r16_10pct_summary["eval_accuracy"],
        "eval_f1_macro": lora_r16_10pct_summary["eval_f1_macro"],
        "train_time_sec": lora_r16_10pct_summary["train_time_sec"],
        "peak_gpu_mem_mb": lora_r16_10pct_summary["peak_gpu_mem_mb"],
        "trainable_pct": lora_r16_10pct_summary["trainable_pct"],
    },
])

print(compare_head_vs_lora_10pct)


## 9.3 Full Fine-Tuning on 10% Split

Run full fine-tuning on the 10% split and create the final 10% comparison table.


In [ ]:
full_ft_10pct_model = build_full_ft_model()
print_parameter_report(full_ft_10pct_model, title="Full fine-tuning model for 10pct run")


In [ ]:
import inspect
from transformers import TrainingArguments

raw_args_full_ft_10pct = dict(
    output_dir="./runs/full_ft_10pct",

    do_train=True,
    do_eval=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,

    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=64,

    logging_strategy="steps",
    logging_steps=100,

    fp16=True,
    bf16=False,

    report_to="none",
    seed=SEED
)

sig = inspect.signature(TrainingArguments.__init__)
accepted = set(sig.parameters.keys())
filtered_args_full_ft_10pct = {k: v for k, v in raw_args_full_ft_10pct.items() if k in accepted}

training_args_full_ft_10pct = TrainingArguments(**filtered_args_full_ft_10pct)
print(training_args_full_ft_10pct)


In [ ]:
full_ft_10pct_trainer = Trainer(
    model=full_ft_10pct_model,
    args=training_args_full_ft_10pct,
    train_dataset=tokenized_train_10pct,
    eval_dataset=tokenized_val_5k,
    data_collator=data_collator,
    compute_metrics=compute_classification_metrics,
)

from transformers.utils.notebook import NotebookProgressCallback

try:
    full_ft_10pct_trainer.remove_callback(NotebookProgressCallback)
    print("NotebookProgressCallback removed")
except Exception as e:
    print("Callback removal note:", e)


In [ ]:
reset_gpu_memory()

start_time = time.time()
full_ft_10pct_trainer.train()
end_time = time.time()

full_ft_10pct_train_time_sec = end_time - start_time
full_ft_10pct_peak_mem_mb = get_peak_gpu_memory_mb()

print("Training finished.")
print(f"Wall-clock training time (sec): {full_ft_10pct_train_time_sec:.2f}")
print(f"Peak GPU memory during training (MB): {full_ft_10pct_peak_mem_mb:.2f}")


In [ ]:
full_ft_10pct_eval = full_ft_10pct_trainer.evaluate()

print("Full FT 10pct eval results:")
for k, v in full_ft_10pct_eval.items():
    print(f"{k}: {v}")


In [ ]:
full_ft_10pct_summary = {
    "method": "full_ft_10pct",
    "train_samples": len(tokenized_train_10pct),
    "val_samples": len(tokenized_val_5k),
    "epochs": training_args_full_ft_10pct.num_train_epochs,
    "trainable_params": count_parameters(full_ft_10pct_model)["trainable_params"],
    "trainable_pct": count_parameters(full_ft_10pct_model)["trainable_pct"],
    "train_time_sec": full_ft_10pct_train_time_sec,
    "peak_gpu_mem_mb": full_ft_10pct_peak_mem_mb,
    "eval_accuracy": full_ft_10pct_eval.get("eval_accuracy"),
    "eval_f1_macro": full_ft_10pct_eval.get("eval_f1_macro"),
    "eval_loss": full_ft_10pct_eval.get("eval_loss"),
}

print(full_ft_10pct_summary)


In [ ]:
final_10pct_df = pd.DataFrame([
    {
        "method": "head_only_10pct",
        "rank": None,
        "eval_accuracy": head_only_10pct_summary["eval_accuracy"],
        "eval_f1_macro": head_only_10pct_summary["eval_f1_macro"],
        "train_time_sec": head_only_10pct_summary["train_time_sec"],
        "peak_gpu_mem_mb": head_only_10pct_summary["peak_gpu_mem_mb"],
        "trainable_pct": head_only_10pct_summary["trainable_pct"],
    },
    {
        "method": lora_r4_10pct_summary["method"],
        "rank": lora_r4_10pct_summary["rank"],
        "eval_accuracy": lora_r4_10pct_summary["eval_accuracy"],
        "eval_f1_macro": lora_r4_10pct_summary["eval_f1_macro"],
        "train_time_sec": lora_r4_10pct_summary["train_time_sec"],
        "peak_gpu_mem_mb": lora_r4_10pct_summary["peak_gpu_mem_mb"],
        "trainable_pct": lora_r4_10pct_summary["trainable_pct"],
    },
    {
        "method": lora_r8_10pct_summary["method"],
        "rank": lora_r8_10pct_summary["rank"],
        "eval_accuracy": lora_r8_10pct_summary["eval_accuracy"],
        "eval_f1_macro": lora_r8_10pct_summary["eval_f1_macro"],
        "train_time_sec": lora_r8_10pct_summary["train_time_sec"],
        "peak_gpu_mem_mb": lora_r8_10pct_summary["peak_gpu_mem_mb"],
        "trainable_pct": lora_r8_10pct_summary["trainable_pct"],
    },
    {
        "method": lora_r16_10pct_summary["method"],
        "rank": lora_r16_10pct_summary["rank"],
        "eval_accuracy": lora_r16_10pct_summary["eval_accuracy"],
        "eval_f1_macro": lora_r16_10pct_summary["eval_f1_macro"],
        "train_time_sec": lora_r16_10pct_summary["train_time_sec"],
        "peak_gpu_mem_mb": lora_r16_10pct_summary["peak_gpu_mem_mb"],
        "trainable_pct": lora_r16_10pct_summary["trainable_pct"],
    },
    {
        "method": full_ft_10pct_summary["method"],
        "rank": None,
        "eval_accuracy": full_ft_10pct_summary["eval_accuracy"],
        "eval_f1_macro": full_ft_10pct_summary["eval_f1_macro"],
        "train_time_sec": full_ft_10pct_summary["train_time_sec"],
        "peak_gpu_mem_mb": full_ft_10pct_summary["peak_gpu_mem_mb"],
        "trainable_pct": full_ft_10pct_summary["trainable_pct"],
    },
])

print(final_10pct_df)


# 10. Save Final Experiment Summaries

Save all available experiment summaries to Google Drive in `summaries.json`.


In [ ]:
import json, os
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/amazon-sentiment"
os.makedirs(SAVE_DIR, exist_ok=True)

summaries = {}
for name, var in [
    # 1%
    ("head_only_1pct",    head_only_1pct_summary),
    ("lora_r4_1pct",      lora_r4_summary),
    ("lora_r8_1pct",      lora_r8_summary),
    ("lora_r16_1pct",     lora_r16_summary),
    ("full_ft_1pct",      full_ft_1pct_summary),
    # 5%
    ("head_only_5pct",    head_only_5pct_summary),
    ("lora_r4_5pct",      lora_r4_5pct_summary),
    ("lora_r8_5pct",      lora_r8_5pct_summary),
    ("lora_r16_5pct",     lora_r16_5pct_summary),
    ("full_ft_5pct",      full_ft_5pct_summary),
    # 10%
    ("head_only_10pct",   head_only_10pct_summary),
    ("lora_r4_10pct",     lora_r4_10pct_summary),
    ("lora_r8_10pct",     lora_r8_10pct_summary),
    ("lora_r16_10pct",    lora_r16_10pct_summary),
    ("full_ft_10pct",     full_ft_10pct_summary),
]:
    try:
        summaries[name] = var
        print(f"  saved: {name}")
    except NameError:
        print(f"  skipped: {name}")

with open(f"{SAVE_DIR}/summaries.json", "w") as f:
    json.dump(summaries, f, indent=2)

print(f"\nDone. {len(summaries)} summaries saved.")
